### 匹配(Match-Case)語句

- 匹配語句的語法在**Python 3.10**版本之後開始出現。
- 匹配語句接受一個表達式做爲匹配對象，並將匹配對象的值與**案例區塊(case)**中的模式進行比較。
- 這在表面上類似於 C、Java 或 JavaScript 中的 switch 語句，但更類似於 Rust 或 Haskell 等語言中的模式匹配。
- 只有第一個匹配的模式會被執行，並且它還可以**從匹配對象中提取或捕獲某個組件(componeent)數值或屬性**。
- 例如捕獲序列(Sequence)中的某一元素或物件的某一屬性）到變數中。

最簡單的形式是將與一個匹配對象或多個**常數(literal)**進行比較：

```python
def response_code(status):
    match status:
        case 400:
            return "錯誤請求"
        case 401:
            return "未授權"
        case 403:
            return "禁止訪問"
        case 404:
            return "未找到"
        case 500:
            return "內部伺服器錯誤"
        case 503:
            return "服務不可用"
        case _:
            return "網路出現問題"
```

注意最後一個case：“變數名” _ 作為**通配符 或 萬用字元(wildcard)**，永遠不會無法匹配。如果沒有案例匹配，則不會執行任何的分支區域。

您可以使用 |（“或”）將幾個**常數(literal)**合併到單一模式中：

```python
case 401 | 403 | 404:
    return "不允許"
```

匹配模式可以看起來像**解包賦值 或 解構賦值(unpack assignment)**，並可以用來綁定變數：

```python
# point 是一個 (x, y) 元組
match point:
    case (0, 0):
        print("原點")
    case (0, y):
        print(f"Y={y}")
    case (x, 0):
        print(f"X={x}")
    case (x, y):
        print(f"X={x}, Y={y}")
    case _:
        raise ValueError("不是一個點")
```

上述範例中第一個模式有兩個常數(literal)，可以被視為對上面顯示的常數模式的擴展。但是接下來的兩個模式結合了一個常數和一個變數，**變數從匹配對象（point）綁定一個值**。第四個模式捕獲兩個值，這使得它在概念上類似於**解包賦值 (x, y) = point**。

如果您使用類別來結構化數據，您可以使用類別名稱後跟一個類似於構造函數的參數清單，但能夠將屬性捕獲到變數中：

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

def where_is(point): # point 是 Point物件
    match point:
        case Point(x=0, y=0):
            print("原點")
        case Point(x=0, y=y):
            print(f"Y={y}")
        case Point(x=x, y=0):
            print(f"X={x}")
        case Point():
            print("其他地方")
        case _:
            print("不是一個點")
```

Note: 也可以使用**位置參數**, 但必須在類別中設置 __match_args__ 特殊屬性來定義模式中屬性的特定位置。如果設置為 (“x”, “y”)，則以下模式都是等效的（並且都將 y 屬性綁定到變數 var）：

```python
Point(1, var)
Point(1, y=var)
Point(x=1, y=var)
Point(y=var, x=1)
```


```python
class Point:
    __match_args__ = ('x', 'y')  # Define attributes for pattern matching

    def __init__(self, x, y):
        self.x = x
        self.y = y

def describe_point(point):
    match point:
        case Point(1, var): #位置參數
            print(f"Point at x=1, y={var}")
        case Point(x=2, y=var):
            print(f"Point at x=2, y={var}")
        case Point(x=var, y=2):
            print(f"Point at x={var}, y=2")
        case Point(y=var, x=3):
            print(f"Point at y={var}, x=3")
        case _:
            print("Not a specific point")

# Example usage
points = [
    Point(1, 5),
    Point(2, 10),
    Point(10, 2),
    Point(3, 15),
    Point(10, 15),
]

for p in points:
    describe_point(p)
```


### 理解模式匹配作為賦值

1. **模式作為賦值**：
   - 當你在 `match-case` 語句中寫模式時，你可以將其視為更複雜的賦值語句。在典型的賦值中，你將值賦給變數，這通常是在表達式的右側。在模式匹配中，你實際上是在做相同的事情，但結構可以更為複雜。

2. **獨立變數名稱**：
   - 當你在模式中使用獨立的變數名稱（如 `var`）時，這些變數會被賦予來自匹配對象的對應值。
   - 例子：
     ```python
     match point:
         case Point(x, y):
             # x 和 y 將從匹配的 Point 的屬性中獲取值
             print(f"Point 的 x={x} 和 y={y}")
     ```
   - 在這種情況下，`x` 和 `y` 是獨立的變數名稱，將從 `Point` 對象中獲取值並進行賦值。

3. **點名(Dotted name)和屬性名稱**：
   - 點名(Dotted name)（例如 `foo.bar`）和屬性名稱（例如 `x=`、`y=`）用於匹配特定屬性，但不會被賦值。相反，它們參考的是被匹配對象的屬性。
   - 例子：
     ```python
     match point:
         case Point(x=1, y=y_var):
             # x 是一個明確的匹配條件，但不被賦值；它是條件
             print(f"Point 的 x=1 和 y={y_var}")
     ```
   - 在這個例子中，`x=1` 是一個條件，必須成立才能匹配成功，而 `y_var` 是一個獨立變數，將從匹配的 `Point` 中獲取 `y` 的對應值。

4. **類別名稱**：
   - 在模式中使用的類別名稱（如 `Point` 在 `case Point(...)` 中）也不會被賦值。它們用於識別被匹配對象的類型。
   - 例子：
     ```python
     match shape:
         case Point(x, y):
             print("這是一個點。")
         case Circle(radius):
             print("這是一個圓。")
     ```
   - 在這裡，`Point` 和 `Circle` 是用來識別 `shape` 是哪種類型對象的類名稱。它們有助於確定數據的結構，但不會被賦值給任何變數。

### 總結

- **將模式視為賦值**：在閱讀 `match-case` 語句中的模式時，可以將其視為賦值的擴展形式，這有助於明白哪些變數將從匹配對象中獲得值。
- **變數綁定**：只有獨立的變數名稱（如 `var`、`x`、`y`）會根據匹配的數據獲得值。
- **不可賦值的元素**：點名、屬性名稱和類名稱用於指定條件或類型，但不會獲得值。

### 範例代碼

```python
class Circle:
    __match_args__ = ('radius',)
    def __init__(self, radius):
        self.radius = radius

class Point:
    __match_args__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

def describe_shape(shape):
    match shape:
        case Point(x, y):  # x 和 y 被賦值
            print(f"Point 位於 x={x}, y={y}")
        case Circle(radius):  # radius 被賦值
            print(f"Circle 的半徑={radius}")
        case _:
            print("未知形狀")

# 示例用法
shapes = [Point(1, 2), Circle(5), "Triangle"]

for shape in shapes:
    describe_shape(shape)

```

### 預期輸出

```
Point 位於 x=1, y=2
Circle 的半徑=5
未知形狀
``` 

- 模式可以任意嵌套

例如，如果我們有一個Point的清單`points`，並添加了 __match_args__，我們可以這樣匹配：

```python
class Point:
    __match_args__ = ('x', 'y')
    def __init__(self, x, y):
        self.x = x
        self.y = y

match points: # 清單points
    case []:
        print("沒有點")
    case [Point(0, 0)]:
        print("原點")
    case [Point(x, y)]:
        print(f"單一點 {x}, {y}")
    case [Point(0, y1), Point(0, y2)]:
        print(f"Y 軸上有兩個點 {y1}, {y2}")
    case _:
        print("其他")
```

- 我們可以向模式添加 if 子句，稱為**守衛(guard)**。

如果守衛為假，則匹配會繼續嘗試下一個案例塊。注意，值捕獲發生在守衛評估之前：

```python
match point:
    case Point(x, y) if x == y:
        print(f"Y=X 在 {x}")
    case Point(x, y):
        print(f"不在對角線上")
```


### 其他關鍵特徵

- 像解包賦值一樣，元組和清單模式具有完全相同的含義，實際上匹配任意序列。一個重要的例外是它們**不匹配迭代器或字串**。

- 序列模式支持擴展解包：[x, y, *rest] 和 (x, y, *rest) 的工作原理類似於解包賦值。* 後面的名稱也可以是 _，因此 (x, y, *_) 匹配至少兩個項目的序列，而不綁定其餘項目。

- 字典(dict)模式：{"bandwidth": b, "latency": l} 從字典中捕獲 "bandwidth" 和 "latency" 值。**與序列模式不同，額外的鍵會被忽略** (序列模式必須匹配相同的元素個數)。也支持像 **rest 這樣的解包。（但是 **_ 將是多餘的，因此不允許）。

- 子模式可以使用 as 關鍵字捕獲：

```python
case (Point(x1, y1), Point(x2, y2) as p2): ...
```

將捕獲輸入的第二個元素為 p2（只要輸入是一個兩個點的序列）。

- 大多數字面量通過相等進行比較，然而單例 True、False 和 None 通過身份進行比較。

- 模式可以使用**名稱常數(Named constants)**。這些必須是**點狀(dotted name)名稱**，以防止它們被解釋為捕獲變數：

```python
from enum import Enum
class Color(Enum):
    RED = 'red'
    GREEN = 'green'
    BLUE = 'blue'

color = Color(input("輸入你選擇的 'red', 'blue' 或 'green': "))

match color:
    case Color.RED: # 必須使用點名Color.RED, 不能使用 RED
        print("我看到紅色！")
    case Color.GREEN:
        print("草是綠色的")
    case Color.BLUE:
        print("我感到憂鬱 :(")
```
